![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 02: AI Modelling Basics)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-ai](https://github.com/tulip-lab/agentic-ai/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 2C: Embeddings, Vector Data and Similarity Search

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Main output</td><td>A small text-vector retrieval system using transparent bag-of-words embeddings</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m02c-overview)
2. [Setup and Background](#m02c-setup)
3. [Core Concepts](#m02c-core-concepts)
4. [Guided Implementation](#m02c-guided-implementation)
5. [Testing and Analysis](#m02c-testing)
6. [Student Tasks](#m02c-student-tasks)
7. [Submission and Reflection](#m02c-submission)

---

<a id="m02c-overview"></a>

### 1. Overview and Learning Goals

This session introduces embeddings, vector data and similarity search. These ideas are central to modern RAG systems, semantic search, recommendation, clustering and many agentic AI workflows.

In `M02A`, a feature was a single numeric value. In `M02B`, an image was represented as a tensor. In this session, a text passage is represented as a vector. A vector is a list of numbers. Once text is represented numerically, we can compare passages using similarity measures such as cosine similarity.

This notebook deliberately uses a transparent bag-of-words embedding rather than an external embedding API. A real embedding model is much more powerful, but it is also less inspectable. Here, the goal is to understand the mechanics: build a vocabulary, convert text into vectors, compare vectors, retrieve top-k documents, and test failure cases.

This session prepares directly for [M03C-Flowise-RAG-Public-Unit-Docs](../../M03-Context-Orchestration/Flowise/M03C-Flowise-RAG-Public-Unit-Docs.md), [M05A-Basic-RAG-System](../../M05-Knowledge-Agents/Jupyter/M05A-Basic-RAG-System.ipynb), and [M05B-RAG-CourseMaterials-Assistant](../../M05-Knowledge-Agents/Jupyter/M05B-RAG-CourseMaterials-Assistant.ipynb). When future practicals need data or documents, use public unit materials first and then public datasets from [tulip-lab/open-data](https://github.com/tulip-lab/open-data).

By the end of this lab, you should be able to explain what an embedding is, build a simple text-vector representation, compute cosine similarity, retrieve the most relevant passages for a query, interpret retrieval scores, and test normal, edge and failure behaviours.

<a id="m02c-setup"></a>

### 2. Setup and Background

This notebook uses only the Python standard library plus `numpy` and `pandas`. It does not call any external embedding API. This keeps the practical reproducible and avoids requiring API keys.

The retrieval corpus is a small set of public unit-style text snippets. Each snippet represents one practical topic in the unit. In a later RAG practical, these snippets would be replaced by actual public unit documents such as `README.md`, `SYLLABUS.md`, notebook markdown cells, Flowise handouts or public datasets from [tulip-lab/open-data](https://github.com/tulip-lab/open-data).

<div align="center">

<table>
<thead>
<tr><th><strong>Concept</strong></th><th><strong>Meaning in this notebook</strong></th><th><strong>Later practical connection</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Embedding</td><td>A numeric vector representing text.</td><td>RAG and semantic search.</td></tr>
<tr><td align="left">Vector store</td><td>A collection of document vectors and metadata.</td><td>Flowise and LangChain retrieval.</td></tr>
<tr><td align="left">Similarity</td><td>A score measuring how close two vectors are.</td><td>Document ranking.</td></tr>
<tr><td align="left">Top-k retrieval</td><td>Return the k most similar documents.</td><td>RAG context selection.</td></tr>
<tr><td align="left">Metadata</td><td>Extra information about each document.</td><td>Source tracing and citation.</td></tr>
</tbody>
</table>

</div>

In [ ]:
import re
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd

print("Setup complete.")

In [ ]:
# Small public unit-style corpus.
# Later RAG practicals can replace this with actual public unit files.

documents = [
    {
        "id": "M01A",
        "title": "Python, Colab and API Safety",
        "text": "Students learn safe API key handling, Python basics, structured outputs, and normal edge failure testing."
    },
    {
        "id": "M01B",
        "title": "GenAI and Agentic AI Fundamentals",
        "text": "Students learn generative AI concepts, large language models, prompts, context, tools, state, and evaluation."
    },
    {
        "id": "M01C",
        "title": "LLMs, Function Calling, APIs and Agent Ecosystems",
        "text": "Students learn LLM API structure, function calling, tool registries, argument validation, and safe routing."
    },
    {
        "id": "M02A",
        "title": "Regression and ML Basics",
        "text": "Students learn supervised learning, features, labels, train test split, linear regression, predictions, and MAE."
    },
    {
        "id": "M02B",
        "title": "Deep Learning Image Classification",
        "text": "Students learn image tensors, synthetic image data, PyTorch CNN models, training loops, accuracy, and prediction inspection."
    },
    {
        "id": "M02C",
        "title": "Embeddings, Vector Data and Similarity Search",
        "text": "Students learn embeddings, vector representations, cosine similarity, top-k retrieval, and retrieval testing."
    },
    {
        "id": "M03C",
        "title": "Flowise RAG over Unit Documents",
        "text": "Students build a visual RAG workflow over public unit documents using document loading, vector stores, retrievers, and chat models."
    },
    {
        "id": "M05A",
        "title": "Basic RAG System",
        "text": "Students build a Python RAG system with chunking, embeddings, vector search, retrieval, prompting, and answer generation."
    },
]

pd.DataFrame(documents)

The table is the miniature corpus. Each row is a document-like item with an `id`, `title` and `text`. A real RAG system may use many more documents and may split each document into smaller chunks. The core idea is the same: each item gets converted into a vector so that it can be compared with a query vector.

<a id="m02c-core-concepts"></a>

### 3. Core Concepts

An embedding is a numeric representation of an object such as a word, sentence, paragraph, image or audio clip. In text retrieval, an embedding should place semantically related texts close to one another in vector space.

In production systems, embeddings are usually generated by trained models. In this notebook, we use a simple bag-of-words embedding. This representation counts how many times vocabulary terms appear in a text. It is less powerful than a neural embedding, but it is transparent.

Cosine similarity compares the direction of two vectors. It is often used for text retrieval because it focuses on relative term patterns rather than raw vector length. The value is usually between 0 and 1 for non-negative bag-of-words vectors.

```text
higher cosine similarity = more similar vector direction
```

<div align="center">

<table>
<thead>
<tr><th><strong>Representation</strong></th><th><strong>Strength</strong></th><th><strong>Limitation</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Bag-of-words</td><td>Simple and inspectable.</td><td>Misses synonyms and deeper meaning.</td></tr>
<tr><td align="left">TF-IDF</td><td>Downweights common words.</td><td>Still mostly lexical.</td></tr>
<tr><td align="left">Neural embedding</td><td>Captures richer semantic similarity.</td><td>Less transparent and may require an external model.</td></tr>
</tbody>
</table>

</div>

In [ ]:
def tokenize(text: str) -> List[str]:
    # Simple tokenizer for teaching purposes.
    return re.findall(r"[a-zA-Z][a-zA-Z\-]*", text.lower())


def build_vocabulary(texts: List[str], min_count: int = 1) -> Dict[str, int]:
    # Build a vocabulary mapping token -> column index.
    if not isinstance(texts, list) or not all(isinstance(t, str) for t in texts):
        raise TypeError("texts must be a list of strings.")
    if min_count <= 0:
        raise ValueError("min_count must be positive.")

    counts = {}
    for text in texts:
        for token in tokenize(text):
            counts[token] = counts.get(token, 0) + 1

    vocab_tokens = sorted([token for token, count in counts.items() if count >= min_count])
    return {token: i for i, token in enumerate(vocab_tokens)}


texts = [doc["text"] for doc in documents]
vocab = build_vocabulary(texts)

print("Vocabulary size:", len(vocab))
print("First 20 vocabulary terms:", list(vocab.keys())[:20])

The vocabulary defines the dimensions of the vector space. If the vocabulary has 80 terms, then each text becomes an 80-dimensional vector. Each column corresponds to one token.

This is a useful simplification: the model does not “understand” language. It only records which vocabulary terms appear. This limitation is why neural embeddings are valuable in real semantic search.

In [ ]:
def embed_text(text: str, vocab: Dict[str, int]) -> Dict[str, Any]:
    # Convert text into a bag-of-words vector.
    if not isinstance(text, str):
        return {"ok": False, "error": "text must be a string.", "result": None}
    if not isinstance(vocab, dict) or len(vocab) == 0:
        return {"ok": False, "error": "vocab must be a non-empty dictionary.", "result": None}

    vector = np.zeros(len(vocab), dtype=float)
    for token in tokenize(text):
        if token in vocab:
            vector[vocab[token]] += 1.0

    return {"ok": True, "error": None, "result": vector}


example_embedding = embed_text(documents[0]["text"], vocab)
print("Embedding ok:", example_embedding["ok"])
print("Vector shape:", example_embedding["result"].shape)
print("Non-zero entries:", int(np.count_nonzero(example_embedding["result"])))

The output shows that a text passage has been converted into a vector. Most entries are zero because each document uses only a subset of the vocabulary. The non-zero entries correspond to words from the document that exist in the vocabulary.

In [ ]:
def cosine_similarity(a: Any, b: Any) -> Dict[str, Any]:
    # Compute cosine similarity with validation.
    try:
        a_arr = np.asarray(a, dtype=float)
        b_arr = np.asarray(b, dtype=float)
    except Exception:
        return {"ok": False, "error": "inputs must be numeric vectors.", "result": None}

    if a_arr.ndim != 1 or b_arr.ndim != 1:
        return {"ok": False, "error": "inputs must be one-dimensional vectors.", "result": None}

    if a_arr.shape != b_arr.shape:
        return {"ok": False, "error": "vectors must have the same shape.", "result": None}

    a_norm = np.linalg.norm(a_arr)
    b_norm = np.linalg.norm(b_arr)

    if a_norm == 0 or b_norm == 0:
        return {"ok": False, "error": "cosine similarity is undefined for zero vectors.", "result": None}

    score = float(np.dot(a_arr, b_arr) / (a_norm * b_norm))
    return {"ok": True, "error": None, "result": score}


v1 = embed_text("RAG uses retrieval and vector search.", vocab)["result"]
v2 = embed_text("Vector search retrieves documents for RAG.", vocab)["result"]
similarity = cosine_similarity(v1, v2)

similarity

The similarity output is a structured dictionary. If `ok=True`, the `result` field stores the cosine similarity score. A higher score means the two vectors have more overlapping term patterns. In this example, both texts mention RAG, retrieval and vector search, so the score should be relatively high.

<a id="m02c-guided-implementation"></a>

### 4. Guided Implementation

We now build a small vector store. A vector store contains document metadata and document vectors. In production systems, vector stores may support approximate nearest-neighbour search, persistence, filtering and metadata-based retrieval. Here, we use a simple in-memory list so that the retrieval logic is visible.

In [ ]:
def build_vector_store(documents: List[Dict[str, str]], vocab: Dict[str, int]) -> Dict[str, Any]:
    # Build a simple in-memory vector store.
    if not isinstance(documents, list) or len(documents) == 0:
        return {"ok": False, "error": "documents must be a non-empty list.", "result": None}

    store = []
    for doc in documents:
        if not all(key in doc for key in ["id", "title", "text"]):
            return {"ok": False, "error": "each document must contain id, title and text.", "result": None}

        emb = embed_text(doc["text"], vocab)
        if not emb["ok"]:
            return emb

        store.append({
            "id": doc["id"],
            "title": doc["title"],
            "text": doc["text"],
            "vector": emb["result"],
        })

    return {"ok": True, "error": None, "result": store}


store_result = build_vector_store(documents, vocab)
vector_store = store_result["result"]

print("Vector store size:", len(vector_store))
print("First item keys:", vector_store[0].keys())

The vector store keeps both the original document information and the vector representation. The original text is still needed because retrieval should return useful context, not only vector IDs.

In [ ]:
def retrieve_top_k(query: str, vector_store: List[Dict[str, Any]], vocab: Dict[str, int], k: int = 3) -> Dict[str, Any]:
    # Retrieve the top-k most similar documents for a query.
    if not isinstance(query, str) or not query.strip():
        return {"ok": False, "error": "query must be a non-empty string.", "result": None}
    if not isinstance(k, int) or k <= 0:
        return {"ok": False, "error": "k must be a positive integer.", "result": None}
    if not isinstance(vector_store, list) or len(vector_store) == 0:
        return {"ok": False, "error": "vector_store must be a non-empty list.", "result": None}

    query_embedding = embed_text(query, vocab)
    if not query_embedding["ok"]:
        return query_embedding

    query_vector = query_embedding["result"]
    if np.linalg.norm(query_vector) == 0:
        return {"ok": False, "error": "query contains no known vocabulary terms.", "result": None}

    scored = []
    for item in vector_store:
        sim = cosine_similarity(query_vector, item["vector"])
        if sim["ok"]:
            scored.append({
                "id": item["id"],
                "title": item["title"],
                "text": item["text"],
                "score": sim["result"],
            })

    scored = sorted(scored, key=lambda x: x["score"], reverse=True)
    return {"ok": True, "error": None, "result": scored[:k]}


query = "How do I build a RAG system with embeddings and vector search?"
retrieval_result = retrieve_top_k(query, vector_store, vocab, k=3)

pd.DataFrame(retrieval_result["result"])

The retrieved results are ranked by similarity score. You should expect RAG-related practicals such as `M05A` or `M03C` to appear near the top, because the query mentions RAG, embeddings and vector search.

The score is not a probability. It is a similarity measure. A result with score `0.60` is not “60% correct”; it simply has a stronger vector match than lower-scoring results under this representation.

In [ ]:
def explain_retrieval(query: str, results: List[Dict[str, Any]]) -> None:
    # Print a readable retrieval explanation.
    print("Query:")
    print(query)
    print()
    print("Top results:")
    for rank, item in enumerate(results, start=1):
        print(f"{rank}. {item['id']} | {item['title']} | score={item['score']:.4f}")
        print(f"   {item['text']}")
        print()


explain_retrieval(query, retrieval_result["result"])

This human-readable explanation is useful for inspection. In a RAG system, the retrieved text would usually be passed as context to an LLM. However, the retrieval step should still be inspectable. If retrieval returns irrelevant documents, the final answer may be weak even if the LLM is strong.

<a id="m02c-testing"></a>

### 5. Testing and Analysis

We now test the embedding and retrieval functions. The tests check normal, edge and failure behaviours.

<div align="center">

<table>
<thead>
<tr><th><strong>Test type</strong></th><th><strong>Purpose</strong></th><th><strong>Example</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Normal</td><td>Check expected retrieval behaviour.</td><td>RAG query retrieves RAG-related documents.</td></tr>
<tr><td align="left">Edge</td><td>Check small but valid inputs.</td><td><code>k=1</code> returns one result.</td></tr>
<tr><td align="left">Failure</td><td>Reject invalid input safely.</td><td>Empty query, unknown vocabulary, invalid k, vector shape mismatch.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Normal case: RAG query should retrieve at least one RAG-related result.
normal_retrieval = retrieve_top_k("RAG retrieval vector search", vector_store, vocab, k=3)
assert normal_retrieval["ok"] is True
top_ids = [item["id"] for item in normal_retrieval["result"]]
assert any(doc_id in top_ids for doc_id in ["M03C", "M05A", "M02C"])

# Edge case: k=1 returns exactly one item.
edge_retrieval = retrieve_top_k("image tensor classification", vector_store, vocab, k=1)
assert edge_retrieval["ok"] is True
assert len(edge_retrieval["result"]) == 1

# Failure case: empty query.
empty_query = retrieve_top_k("", vector_store, vocab, k=3)
assert empty_query["ok"] is False

# Failure case: query with no known vocabulary terms.
unknown_query = retrieve_top_k("zzzz qqqq xxxx", vector_store, vocab, k=3)
assert unknown_query["ok"] is False

# Failure case: invalid k.
bad_k = retrieve_top_k("RAG", vector_store, vocab, k=0)
assert bad_k["ok"] is False

# Failure case: vector shape mismatch.
shape_mismatch = cosine_similarity(np.array([1, 2]), np.array([1, 2, 3]))
assert shape_mismatch["ok"] is False

print("Embedding and retrieval tests passed.")

If this cell prints `Embedding and retrieval tests passed.`, the retrieval system satisfies the behaviours we explicitly checked. This does not make it a production retrieval system. It means the core mechanics and safe rejection behaviours are working for this controlled example.

<a id="m02c-student-tasks"></a>

### 6. Student Tasks

Extend the retrieval corpus and test retrieval behaviour. You should add at least three new public unit-style documents or synthetic document snippets. Do not use private files, student submissions, credentials or unpublished assessment materials.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What you need to do</strong></th><th><strong>Why it matters</strong></th><th><strong>Expected evidence</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1</td><td>Add at least three new document snippets.</td><td>Practises corpus expansion.</td><td>New documents include id, title and text.</td></tr>
<tr><td align="left">Task 2</td><td>Rebuild the vocabulary and vector store.</td><td>Vector dimensions depend on the vocabulary.</td><td>Vocabulary size and store size are shown.</td></tr>
<tr><td align="left">Task 3</td><td>Run at least three queries.</td><td>Practises similarity search and result interpretation.</td><td>Top-k tables or printed explanations.</td></tr>
<tr><td align="left">Task 4</td><td>Write a short analysis of one good result and one weak result.</td><td>Retrieval quality must be inspected, not assumed.</td><td>150 to 250 words.</td></tr>
</tbody>
</table>

</div>

Suggested topics for new snippets:

- Flowise chatbot setup;
- LangChain tools;
- LangGraph stateful workflow;
- private agents with Ollama;
- model evaluation with Hugging Face;
- productised AI agents.

In [ ]:
# Student task starter.
# Add at least three new public unit-style snippets.

student_documents = documents + [
    # TODO: replace or extend these examples.
    {
        "id": "M04B",
        "title": "LangChain Tool-Using Agents",
        "text": "Students build LangChain agents that use approved tools, validate inputs, and return structured outputs."
    },
    {
        "id": "M05C",
        "title": "LangGraph Stateful Workflows",
        "text": "Students build stateful workflows with graph nodes, edges, memory, routing, and controlled execution."
    },
    {
        "id": "M06C",
        "title": "Private Agents with Ollama",
        "text": "Students explore private local agents using open-source models, Ollama, local inference, and safe deployment."
    },
]

# TODO: rebuild vocabulary and vector store.
# student_vocab = build_vocabulary([doc["text"] for doc in student_documents])
# student_store = build_vector_store(student_documents, student_vocab)["result"]

# TODO: run at least three queries and inspect results.

<a id="m02c-submission"></a>

### 7. Submission and Reflection

Submit the completed notebook with the following evidence.

<div align="center">

<table>
<thead>
<tr><th><strong>Required item</strong></th><th><strong>What to submit</strong></th><th><strong>Quality check</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Extended corpus</td><td>At least three new snippets.</td><td>No private or sensitive content.</td></tr>
<tr><td align="left">Rebuilt vocabulary and vector store</td><td>Vocabulary size and vector store size.</td><td>Store includes original and new documents.</td></tr>
<tr><td align="left">Retrieval outputs</td><td>At least three top-k retrieval examples.</td><td>Results are displayed and interpreted.</td></tr>
<tr><td align="left">Tests</td><td>Evidence that provided tests pass.</td><td>Normal, edge and failure behaviours are preserved.</td></tr>
<tr><td align="left">Reflection</td><td>150 to 250 words.</td><td>Reflection explains strengths and limitations of bag-of-words retrieval.</td></tr>
</tbody>
</table>

</div>

Reflection questions:

1. What does an embedding represent in this notebook?
2. Why does vocabulary choice affect the vector representation?
3. What does cosine similarity measure?
4. Why is a high similarity score not the same as a verified answer?
5. How does this retrieval system prepare you for later RAG practicals?

#### Further Readings

- scikit-learn feature extraction: <https://scikit-learn.org/stable/modules/feature_extraction.html>
- LangChain retrieval documentation: <https://python.langchain.com/docs/concepts/retrievers/>
- Flowise vector store documentation: <https://docs.flowiseai.com/>
- Public data repository for this unit: <https://github.com/tulip-lab/open-data>